# **Import Necessary Libraries**

In [ ]:
import torch
import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer

# **Load Dataset**

In [3]:
df = pd.read_csv('D:\\Research Work\\NER\\Dataset\\Barishal_NER.csv')

In [ ]:
df.head(65)

In [ ]:
print(df.groupby('Sentence #').size())  # Check the number of words per sentence

In [ ]:
df['Sentence #'].fillna(method='ffill', inplace=True)

In [ ]:
df.head(65)

In [ ]:
print(df.groupby('Sentence #').size())

# **Dataset Preprocessing**

In [ ]:
# Group words and tags by sentence
sentences = df.groupby('Sentence #')['barishal_word'].apply(list).reset_index(name='words')
tags = df.groupby('Sentence #')['bio_tag'].apply(list).reset_index(name='tags')
print(tags)

In [ ]:
# Combine words and tags into a single DataFrame
data = pd.merge(sentences, tags, on='Sentence #')

# Create a mapping from tags to IDs
unique_tags = set(tag for sublist in data['tags'] for tag in sublist)
tag2id = {tag: id for id, tag in enumerate(unique_tags)}
id2tag = {id: tag for tag, id in tag2id.items()}
print(unique_tags)

# Convert tags to IDs
data['tags'] = data['tags'].apply(lambda x: [tag2id[tag] for tag in x])

# Split the dataset into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)

In [ ]:
# Print the tag-to-ID mapping
print("Tag to ID Mapping (tag2id):")
print(tag2id)

# Print the ID-to-tag mapping
print("\nID to Tag Mapping (id2tag):")
print(id2tag)

# **BERT Base Multilingual Cased Tokenizer**

In [12]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased', use_fast=True)  # Use fast tokenizer

In [ ]:
# Tokenize the dataset with padding and truncation
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['words'],
        truncation=True,
        padding='max_length',  # Pad to the maximum length
        is_split_into_words=True,
        max_length=128,  # Set a maximum length for truncation
    )
    labels = []
    for i, label in enumerate(examples['tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Special token for padding
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])  # Assign the same label to subwords
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

train_tokenized = train_dataset.map(tokenize_and_align_labels, batched=True)
test_tokenized = test_dataset.map(tokenize_and_align_labels, batched=True)

# **BERT Base Multilingual Cased Training**

In [ ]:
# Load BERT Base Multilingual Cased model for token classification
model = AutoModelForTokenClassification.from_pretrained("bert-base-multilingual-cased", num_labels=len(unique_tags))

In [15]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='D:\\Research Work\\NER\\Checkpoints\\results',
    eval_strategy='epoch',  # Use 'eval_strategy' instead of 'evaluation_strategy'
    save_strategy='epoch',  # Align save strategy with eval strategy
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,  # Ensure this matches the eval and save strategies
    metric_for_best_model='f1',
)

# Load evaluation metric
seqeval_metric = evaluate.load('seqeval')

# **Evaluation Matrics**

In [17]:
# Define evaluation metrics
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [[id2tag[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2tag[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

In [ ]:
# Evaluate the model
results = trainer.evaluate()
print(results)

# **Model Save**

In [ ]:
# Save the model
trainer.save_model('D:\\Research Work\\NER\\Model\\Bert_Base_Multilingual_Cased_NER_Model_Barishal_Epoch05')
tokenizer.save_pretrained('D:\\Research Work\\NER\\Model\\Bert_Base_Multilingual_Cased_NER_Model_Barishal_Epoch05')

# **Random Sentence Prediction**

In [ ]:
# Load the trained model and tokenizer
model = AutoModelForTokenClassification.from_pretrained('D:\\Research Work\\NER\\Model\\Bert_Base_Multilingual_Cased_NER_Model_Barishal_Epoch05')
tokenizer = AutoTokenizer.from_pretrained('D:\\Research Work\\NER\\Model\\Bert_Base_Multilingual_Cased_NER_Model_Barishal_Epoch05')

# Function to predict tags for a new sentence
def predict_tags(sentence):
    # Tokenize the input sentence
    inputs = tokenizer(
        sentence.split(),  # Split the sentence into words
        is_split_into_words=True,  # Indicate that the input is already split into words
        truncation=True,  # Truncate to the model's max length
        padding=True,  # Pad to the model's max length
        return_tensors='pt',  # Return PyTorch tensors
        max_length=128,  # Set a maximum length for truncation
    )

    # Get predictions from the model
    with torch.no_grad():
        logits = model(**inputs).logits

    # Get the predicted tag IDs
    predicted_tag_ids = torch.argmax(logits, dim=2).squeeze().tolist()

    # Convert word IDs to tokens and align predictions
    word_ids = inputs.word_ids()  # Get word IDs for each token
    previous_word_idx = None
    predicted_tags = []
    for word_idx, tag_id in zip(word_ids, predicted_tag_ids):
        if word_idx is None or word_idx == previous_word_idx:
            # Skip special tokens (e.g., [CLS], [SEP]) and subword tokens
            continue
        predicted_tags.append(id2tag[tag_id])  # Convert tag ID to tag label
        previous_word_idx = word_idx

    return predicted_tags

# Example sentence
example_sentence = "লাল গরু"

# Predict tags for the example sentence
predicted_tags = predict_tags(example_sentence)

# Print the results
print("Sentence:", example_sentence)
print("Predicted Tags:", predicted_tags)